# 00 · Boiling point: acquire NIST data

![NIST Chemistry WebBook source card with ethanol structure and its tabulated boiling point in kelvin](../docs/data_sources/nist_webbook_boiling_point.png)

*Figure 1. [NIST Chemistry WebBook, SRD 69](https://webbook.nist.gov/chemistry/), illustrated by its [ethanol phase-change record](https://webbook.nist.gov/cgi/cbook.cgi?ID=C64175&Units=SI&Mask=4). This notebook collects boiling-point records in kelvin and molecular identities for conversion to SMILES. Source card uses the official WebBook icon, molecular image and tabulated boiling point; workshop layout, October 21, 2026. Credit: NIST.*

This provenance notebook is not run live in the workshop. It is the compact, current NistChemPy 2.0 workflow used to query the NIST Chemistry WebBook, collect phase-change records, and create a new table of canonical SMILES and boiling points.

The NIST phase-change table reports `T_boil` in kelvin. The final notebook step validates the scraped kelvin records, canonicalizes and deduplicates SMILES, and writes a two-column `SMILES`/`BP` parquet in kelvin for the downstream workshop notebooks. The raw kelvin collection is retained separately for provenance.

**Normal use:** leave live collection disabled. The notebook will reuse an existing `outputs/nist_acquisition/NIST_BP_SMILES.parquet` from the earlier collection and cleanse it without querying NIST again. Enable collection only for a deliberate new crawl.


### Reading and presenting this notebook

Each code block has a descriptive heading followed by a short explanation labeled **Code Summary:**; saved outputs remain visible. In **VS Code**, open the Command Palette (`Ctrl+Shift+P` on Windows/Linux, `Cmd+Shift+P` on macOS) and choose **Notebook: Collapse All Cell Inputs** (search for `Collapse All Cell Inputs`). Use **Expand All Cell Inputs** to edit, or expand an individual cell using its input control. Do not collapse outputs. VS Code can remember your previous view, so use the command if code is still shown. In **JupyterLab**, code inputs are initially hidden via notebook metadata; expand them with the left-hand cell collapser or the View menu. Hiding code does not disable execution or remove it from the file.


### Configure NIST data acquisition

Code Summary: Sets up NIST access, request delays, and resumable data files. Live collection is off by default so existing Parquet data can be reused.


In [8]:
from pathlib import Path
import re
import time

import nistchempy as nist
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from rdkit import Chem
from tqdm.auto import tqdm

OUTPUT_DIR = Path('outputs/nist_acquisition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = OUTPUT_DIR / 'NIST_BP_SMILES.parquet'  # Cleansed Celsius data used by workshop notebooks.
RAW_OUTPUT = OUTPUT_DIR / 'NIST_BP_SMILES_raw_kelvin.parquet'
PROGRESS = OUTPUT_DIR / 'bp_completed_nist_ids.txt'

RUN_LIVE_COLLECTION = False  # Reuse the existing parquet; set True only for a deliberate new multi-hour crawl.
MAX_COMPOUNDS = None  # Set to None only for an intentional full crawl.
CHECKPOINT_EVERY = 25
request_config = nist.RequestConfig(delay=3.0, max_attempts=3, kwargs={'timeout': 30.0})

if nist.__version__ != '2.0.0':
    raise RuntimeError(f'Tested with nistchempy 2.0.0; found {nist.__version__}.')
print(f'NistChemPy {nist.__version__}; live collection: {RUN_LIVE_COLLECTION}')


NistChemPy 2.0.0; live collection: True


### Find compounds with phase-change data

Code Summary: Finds neutral compounds with phase-change data and saves their NIST identifiers for collection. Large searches are subdivided to avoid truncated results; this runs only in live mode.


In [4]:
# Current NistChemPy 2.0 search: neutral compounds advertising phase-change data.
# The formula subdivisions avoid WebBook result truncation.
search_parameters = nist.NistSearchParameters(no_ion=True, allow_other=False, allow_extra=False, cTP=True)

def discover_compound_ids(max_carbons=19):
    identifiers = set()
    for carbon_count in tqdm(range(1, max_carbons + 1), desc='Discovering NIST records'):
        result = nist.run_search(f'C{carbon_count}H*O*N*S*Cl*', 'formula',
                                 search_parameters=search_parameters, request_config=request_config)
        if result.success:
            identifiers.update(result.compound_ids)
        if result.lost:
            for hydrogen_count in range(50):
                narrowed = nist.run_search(f'C{carbon_count}H{hydrogen_count}O*N*S*Cl*', 'formula',
                                           search_parameters=search_parameters, request_config=request_config)
                if narrowed.success:
                    identifiers.update(narrowed.compound_ids)
    return sorted(identifiers)

# Run this only as part of a deliberate new collection. Save the ID list for provenance.
if RUN_LIVE_COLLECTION:
    compound_ids = discover_compound_ids()
    pd.DataFrame({'NIST_molecule': compound_ids}).to_parquet(OUTPUT_DIR / 'bp_candidate_ids.parquet', index=False)
    print(f'Discovered {len(compound_ids):,} candidate records.')


Discovering NIST records: 100%|██████████| 19/19 [36:47<00:00, 116.20s/it]


Discovered 12,296 candidate records.


### Extract SMILES and kelvin measurements

Code Summary: Defines how to extract canonical SMILES and boiling-point measurements reported in kelvin. An illustrative compound is fetched from NIST, so this cell requires internet access.


In [5]:
def extract_boiling_point_kelvin(phase_change_url):
    """Return a NIST T_boil value only when the table explicitly reports kelvin."""
    if not phase_change_url:
        return None
    response = requests.get(phase_change_url, timeout=30)
    response.raise_for_status()
    time.sleep(request_config.delay)
    for row in BeautifulSoup(response.text, 'html.parser').select('tr'):
        cells = row.find_all(['th', 'td'])
        if len(cells) < 2:
            continue
        label = re.sub(r'[^a-z]', '', cells[0].get_text(' ', strip=True).lower())
        if 'tboil' not in label and 'boiling' not in label:
            continue
        units = cells[2].get_text(' ', strip=True) if len(cells) >= 3 else ''
        if units.casefold() != 'k':
            raise ValueError(f'Expected kelvin T_boil units from NIST, found {units!r}')
        number = re.search(r'[-+]?\d*\.?\d+', cells[1].get_text(' ', strip=True))
        if number:
            return float(number.group())
    return None

def record_from_nist(compound_id):
    compound = nist.get_compound(compound_id, request_config=request_config)
    if compound is None:
        raise LookupError('No unique NIST record.')
    phase_change_url = compound.data_refs.get('cTP')
    boiling_point = extract_boiling_point_kelvin(phase_change_url)
    if boiling_point is None:
        raise LookupError('No explicitly labeled boiling point.')
    molblock = compound.get_mol2D()
    molecule = Chem.MolFromMolBlock(molblock, sanitize=True, removeHs=True)
    if molecule is None:
        raise ValueError('NIST 2-D molfile could not be converted to RDKit.')
    return {'SMILES': Chem.MolToSmiles(molecule, canonical=True), 'BP_K': boiling_point}

# One-record API and schema test. This confirms the current NistChemPy 2 workflow.
example = record_from_nist('C106365')
print(example)


{'SMILES': 'CCCOC(=O)CC', 'BP': 396.0}


### Collect or reload raw boiling points

Code Summary: Collects missing measurements with checkpoints and a failure log, or reloads the existing raw dataset. Original SMILES and kelvin measurements are preserved before cleaning.


In [10]:
if RUN_LIVE_COLLECTION:
    candidate_file = OUTPUT_DIR / 'bp_candidate_ids.parquet'
    if not candidate_file.exists():
        raise FileNotFoundError('Run discovery first or provide bp_candidate_ids.parquet.')
    candidate_ids = pd.read_parquet(candidate_file)['NIST_molecule'].tolist()
    if MAX_COMPOUNDS is not None:
        candidate_ids = candidate_ids[:MAX_COMPOUNDS]
    # Migrate a pre-cleansing collection from the previous notebook version once.
    if not RAW_OUTPUT.exists() and OUTPUT.exists():
        legacy = pd.read_parquet(OUTPUT)
        if set(legacy.columns) != {'SMILES', 'BP'}:
            raise ValueError(f'Cannot migrate legacy output with columns: {legacy.columns.tolist()}')
        legacy.rename(columns={'BP': 'BP_K'}).to_parquet(RAW_OUTPUT, index=False)
    records = pd.read_parquet(RAW_OUTPUT).to_dict('records') if RAW_OUTPUT.exists() else []
    completed = set(PROGRESS.read_text(encoding='utf-8').splitlines()) if PROGRESS.exists() else set()
    failures = []
    for count, compound_id in enumerate(tqdm(candidate_ids, desc='Collecting BP and SMILES'), start=1):
        if compound_id in completed:
            continue
        try:
            records.append(record_from_nist(compound_id)); completed.add(compound_id)
        except Exception as error:
            failures.append({'NIST_molecule': compound_id, 'error': repr(error)})
        if count % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records, columns=['SMILES', 'BP_K']).dropna().drop_duplicates('SMILES').to_parquet(RAW_OUTPUT, index=False)
            PROGRESS.write_text('\n'.join(sorted(completed)), encoding='utf-8')
    result = pd.DataFrame(records, columns=['SMILES', 'BP_K']).dropna().drop_duplicates('SMILES')
    result.to_parquet(RAW_OUTPUT, index=False)
    PROGRESS.write_text('\n'.join(sorted(completed)), encoding='utf-8')
    pd.DataFrame(failures).to_csv(OUTPUT_DIR / 'bp_failures.csv', index=False)
    print(f'Wrote {len(result):,} raw kelvin records to {RAW_OUTPUT}.')
else:
    print('Live collection disabled. Set RUN_LIVE_COLLECTION=True outside the workshop.')


[01:51:13] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 3 ignored.
[01:51:13] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 5 ignored.
[01:51:13] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 3 ignored.

Wrote 4,660 records to outputs\nist_acquisition\NIST_BP_SMILES.parquet.


## Final data examination and cleansing

The raw collection is retained in kelvin. This final stage accepts only finite, physically plausible kelvin values; re-canonicalizes valid SMILES; merges repeated structures with the median temperature; and records the full audit separately. The workshop parquet contains only `SMILES` and `BP`, with `BP` in K.


### Clean and export boiling-point data

Code Summary: Cleans cached measurements without downloading them again, checks molecular structures and plausible temperatures, and combines repeated structures by median boiling point. Saves a two-column SMILES/BP table in kelvin and an audit; range checks alone cannot confirm ambiguous source units.


In [15]:
from pathlib import Path

import pandas as pd
from rdkit import Chem

OUTPUT_DIR = Path('outputs/nist_acquisition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = OUTPUT_DIR / 'NIST_BP_SMILES.parquet'
RAW_OUTPUT = OUTPUT_DIR / 'NIST_BP_SMILES_raw_kelvin.parquet'
MIN_PLAUSIBLE_BP_K = 1.0
MAX_PLAUSIBLE_BP_K = 2000.0
AUDIT_FILE = OUTPUT_DIR / 'NIST_BP_SMILES_cleaning_audit.csv'

# Support one-time cleaning of an output made by the preceding notebook version.
if not RAW_OUTPUT.exists() and OUTPUT.exists():
    legacy = pd.read_parquet(OUTPUT)
    if set(legacy.columns) != {'SMILES', 'BP'}:
        raise ValueError(f'Cannot migrate legacy output with columns: {legacy.columns.tolist()}')
    legacy.rename(columns={'BP': 'BP_K'}).to_parquet(RAW_OUTPUT, index=False)
if not RAW_OUTPUT.exists():
    raise FileNotFoundError(f'No raw NIST collection found at {RAW_OUTPUT}. Run the collection cell first.')

raw = pd.read_parquet(RAW_OUTPUT).copy()
if set(raw.columns) != {'SMILES', 'BP_K'}:
    raise ValueError(f'Raw NIST table must contain only SMILES and BP_K; found {raw.columns.tolist()}')
raw['BP_K'] = pd.to_numeric(raw['BP_K'], errors='coerce')

def canonicalize_smiles(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    molecule = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(molecule, canonical=True) if molecule is not None else None

raw['canonical_SMILES'] = raw['SMILES'].map(canonicalize_smiles)
raw['valid_kelvin'] = raw['BP_K'].between(MIN_PLAUSIBLE_BP_K, MAX_PLAUSIBLE_BP_K, inclusive='both')
raw['valid_smiles'] = raw['canonical_SMILES'].notna()
raw['keep'] = raw['valid_kelvin'] & raw['valid_smiles']

valid = raw.loc[raw['keep'], ['canonical_SMILES', 'BP_K']].copy()
valid['BP'] = valid['BP_K']
grouped = valid.groupby('canonical_SMILES', as_index=False).agg(
    BP=('BP', 'median'), source_records=('BP', 'size'), min_BP_K=('BP', 'min'), max_BP_K=('BP', 'max')
)
cleaned = grouped.rename(columns={'canonical_SMILES': 'SMILES'})[['SMILES', 'BP']].sort_values('SMILES').reset_index(drop=True)
if cleaned.empty or not cleaned.BP.between(MIN_PLAUSIBLE_BP_K, MAX_PLAUSIBLE_BP_K).all():
    raise ValueError('Cleansing produced an empty or physically implausible kelvin target table.')

audit = pd.DataFrame([{
    'raw_records': len(raw), 'invalid_smiles_removed': int((~raw['valid_smiles']).sum()),
    'invalid_kelvin_removed': int((~raw['valid_kelvin']).sum()), 'records_after_validation': len(valid),
    'duplicate_structures_merged': int(len(valid) - len(cleaned)), 'clean_records': len(cleaned),
    'units': 'K (verified from the NIST T_boil units column)',
}])
cleaned.to_parquet(OUTPUT, index=False)
audit.to_csv(AUDIT_FILE, index=False)
print(f'Wrote {len(cleaned):,} cleansed records in K to {OUTPUT}.')
print(f'Wrote cleaning audit to {AUDIT_FILE}.')


Wrote 4,659 cleansed records in K to outputs\nist_acquisition\NIST_BP_SMILES.parquet.
Wrote cleaning audit to outputs\nist_acquisition\NIST_BP_SMILES_cleaning_audit.csv.


## Handoff

`01_BP_Chemprop_GNN.ipynb` uses `outputs/nist_acquisition/NIST_BP_SMILES.parquet`, the cleansed kelvin table produced above. The raw kelvin table and the cleaning audit remain in the same directory for provenance.
